<a href="https://colab.research.google.com/github/shilpitha-03/VideoRAG/blob/hf-embedding-fix/videorag_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted at /content/drive/MyDrive/")

Mounted at /content/drive
✓ Drive mounted at /content/drive/MyDrive/


In [ ]:
import torch
import subprocess

print("=== CUDA state after each import ===")

print(f"Baseline: {torch.cuda.is_initialized()}")

import os
print(f"After os: {torch.cuda.is_initialized()}")

import sys
print(f"After sys: {torch.cuda.is_initialized()}")

# Check nvidia-smi without touching torch
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used',
                     '--format=csv,noheader'],
                    capture_output=True, text=True)
print(f"GPU: {gpu.stdout.strip()}")
print(f"After nvidia-smi: {torch.cuda.is_initialized()}")

=== CUDA state after each import ===
Baseline: False
After os: False
After sys: False
GPU: NVIDIA A100-SXM4-80GB, 0 MiB
After nvidia-smi: False


Cell 2 — Runtime verificationRun this first. Confirms you got the A100 and high RAM before spending time on anything else.

In [ ]:
import subprocess
import os

# Verify GPU
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu_info.stdout)

# Verify RAM
ram_info = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(ram_info.stdout)

# Verify disk space on local disk
disk_info = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print(disk_info.stdout)

if os.path.exists('/content/drive/MyDrive'):
    print("✓ Drive confirmed mounted")
else:
    print("✗ Drive NOT mounted - run Cell 1 first")

Thu May 14 18:39:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

You want to see: A100 40GB in the GPU output, 80GB+ RAM, and 100GB+ free on /content/. If you see a T4 instead, go to Runtime → Change runtime type and select A100 + High RAM before continuing.

Cell 3 — Define all paths

In [ ]:
import os

if not os.path.exists('/content/drive/MyDrive'):
    raise RuntimeError("Drive not mounted - run Cell 1 first")
print("\n✓ Drive confirmed mounted, all paths ready.")

# Drive paths - persistent across sessions
drive_root = '/content/drive/MyDrive/videorag_project'
drive_paths = {
    'trimmed_videos': f'{drive_root}/trimmed_videos',
    'workdir':        f'{drive_root}/workdir',
    'inspection':     f'{drive_root}/inspection_outputs',
}

# Local disk paths - temporary, recreated each session
local_paths = {
    'weights':    '/content/model_weights',
    'whisper':    '/content/model_weights/faster-distil-whisper-large-v3',
    'minicpm':    '/content/model_weights/MiniCPM-V-2_6-int4',
    'imagebind':  '/content/model_weights/imagebind_huge.pth',
    'raw_videos': '/content/raw_videos',
    'cache':      '/content/videorag_cache',
}

# Create Drive folders
print("=== Drive folders (persistent) ===")
for name, path in drive_paths.items():
    os.makedirs(path, exist_ok=True)
    print(f"✓ {name}: {path}")

# Create local folders (.pth is a file not a folder, skip it)
print("\n=== Local folders (this session only) ===")
for name, path in local_paths.items():
    if not path.endswith('.pth'):
        os.makedirs(path, exist_ok=True)
        print(f"✓ {name}: {path}")

print("\nAll paths ready.")


✓ Drive confirmed mounted, all paths ready.
=== Drive folders (persistent) ===
✓ trimmed_videos: /content/drive/MyDrive/videorag_project/trimmed_videos
✓ workdir: /content/drive/MyDrive/videorag_project/workdir
✓ inspection: /content/drive/MyDrive/videorag_project/inspection_outputs

=== Local folders (this session only) ===
✓ weights: /content/model_weights
✓ whisper: /content/model_weights/faster-distil-whisper-large-v3
✓ minicpm: /content/model_weights/MiniCPM-V-2_6-int4
✓ raw_videos: /content/raw_videos
✓ cache: /content/videorag_cache

All paths ready.


Every cell from here references drive_paths or local_paths. One place to change if anything moves.

Cell 4 — Clone repo and verify branch

In [ ]:
import os

# Clone your fork - only runs if not already cloned
if not os.path.exists('/content/VideoRAG'):
    !git clone https://github.com/shilpitha-03/VideoRAG.git /content/VideoRAG

# Change working directory permanently for this session
%cd /content/VideoRAG/VideoRAG-algorithm

# Checkout the branch with the HuggingFace embedding fix
!git checkout hf-embedding-fix

# Show current branch and recent commits
!git branch
!git log --oneline -3

# Actively verify the code change is present
# If this prints ✗, the wrong branch is checked out
!grep -n "huggingface.co" videorag/_llm.py \
    && echo "✓ HuggingFace endpoint confirmed in _llm.py" \
    || echo "✗ Change not found - wrong branch or push failed"

Cloning into '/content/VideoRAG'...
remote: Enumerating objects: 616, done.
remote: Counting objects: 100% (340/340), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 616 (delta 217), reused 177 (delta 163), pack-reused 276 (from 2)
Receiving objects: 100% (616/616), 6.70 MiB | 16.74 MiB/s, done.
Resolving deltas: 100% (302/302), done.
/content/VideoRAG/VideoRAG-algorithm
Branch 'hf-embedding-fix' set up to track remote branch 'hf-embedding-fix' from 'origin'.
Switched to a new branch 'hf-embedding-fix'
* hf-embedding-fix
  main
33ec1fb (HEAD -> hf-embedding-fix, origin/hf-embedding-fix) restore .cuda() in caption.py subprocess - kept consistent with full precision model approach
7a8b779 use full precision MiniCPM-V with .cuda() - switching from int4 to full model; int4 requires bitsandbytes 0.43.1 which has no CUDA 12.8 binary making it incompatible with Colab A100
b5d9e75 load MiniCPM-V exactly as model card specifies - no device_map, no .cuda(), just from_pretr

In [ ]:
%cd /content/VideoRAG/VideoRAG-algorithm
!git pull origin hf-embedding-fix
!git log --oneline -3

/content/VideoRAG/VideoRAG-algorithm
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 11 (delta 9), reused 11 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 1.17 KiB | 399.00 KiB/s, done.
From https://github.com/shilpitha-03/VideoRAG
 * branch            hf-embedding-fix -> FETCH_HEAD
   b5d9e75..33ec1fb  hf-embedding-fix -> origin/hf-embedding-fix
Updating b5d9e75..33ec1fb
Fast-forward
 VideoRAG-algorithm/videorag/_videoutil/caption.py | 10 +++++---
 VideoRAG-algorithm/videorag/videorag.py           | 29 ++++++++++++++++-------
 2 files changed, 28 insertions(+), 11 deletions(-)
33ec1fb (HEAD -> hf-embedding-fix, origin/hf-embedding-fix) restore .cuda() in caption.py subprocess - kept consistent with full precision model approach
7a8b779 use full precision MiniCPM-V with .cuda() - switching from int4 to full model; int4 requires bitsandbytes 0.43.1 which has no CUDA 12.

Cell 5a — Install dependencies

In [ ]:
# Install PyTorch - no version pin, resolves for this Colab environment
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

import torch
print(f"PyTorch: {torch.__version__}")
# Do NOT call torch.cuda.is_available() or any cuda function here
# Any torch.cuda call initializes CUDA which breaks multiprocessing fork in Cell 10
print("✓ PyTorch installed - CUDA will initialize lazily when first needed by a model")

PyTorch: 2.10.0+cu128
✓ PyTorch installed - CUDA will initialize lazily when first needed by a model


Cell 5b — Rest of dependencies

In [ ]:
!pip install accelerate -q
!pip install bitsandbytes -q
!pip install moviepy==1.0.3 -q
!pip install timm ftfy regex einops fvcore eva-decord==0.6.1 iopath -q
!pip install ctranslate2==4.4.0 faster_whisper==1.0.3 -q
!pip install hnswlib xxhash nano-vectordb neo4j -q
# !pip install transformers -q
!pip install transformers==4.43.3 -q
!pip install tiktoken openai tenacity -q
!pip install yt-dlp -q
!pip install ollama==0.5.3 -q

!pip install --no-deps \
    git+https://github.com/facebookresearch/pytorchvideo.git@28fe037d212663c6a24f373b94cc5d478c8c1a1d -q
!pip install --no-deps \
    git+https://github.com/facebookresearch/ImageBind.git@3fcf5c9039de97f6ff5528ee4a9dce903c5979b3 -q

print("✓ All dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 137.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 127.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 31.4 MB/s et

In [ ]:
import importlib
import transformers
importlib.reload(transformers)
print(transformers.__version__)

4.43.3


5c-patch the broken file in place

In [ ]:
# Patch pytorchvideo's broken import of functional_tensor
# This module was removed in torchvision 0.17 (torch 2.2+)
# pytorchvideo hasn't been updated to handle this
# The patch creates a compatibility shim in place of the missing module

import torchvision.transforms.functional as F
import sys
import types

# Create a fake functional_tensor module with the functions
# pytorchvideo actually uses from it
fake_module = types.ModuleType('torchvision.transforms.functional_tensor')

# These are the specific functions pytorchvideo imports from functional_tensor
# They still exist in torchvision.transforms.functional under the same names
funcs_to_copy = [
    'rgb_to_grayscale',
    'adjust_brightness',
    'adjust_contrast',
    'adjust_saturation',
    'adjust_hue',
]

for func_name in funcs_to_copy:
    if hasattr(F, func_name):
        setattr(fake_module, func_name, getattr(F, func_name))

# Register the fake module so pytorchvideo's import finds it
sys.modules['torchvision.transforms.functional_tensor'] = fake_module

print("✓ pytorchvideo compatibility patch applied")

# Verify the patch works by importing what was failing
try:
    from pytorchvideo.transforms import augmix
    print("✓ pytorchvideo imports successfully after patch")
except Exception as e:
    print(f"✗ Patch incomplete: {e}")

✓ pytorchvideo compatibility patch applied
✓ pytorchvideo imports successfully after patch


 Cell 5d Create a symlink that makes cuDNN 8 available where the system expects it.

In [ ]:
import subprocess
import os

# ctranslate2 bundles cuDNN 8 internally
# bitsandbytes looks for libcudnn_ops_infer.so.8 in system paths
# Solution: symlink ctranslate2's bundled cuDNN 8 to where bitsandbytes looks

cudnn8_source = '/usr/local/lib/python3.12/dist-packages/ctranslate2.libs/libcudnn-463fd6d5.so.8.9.7'

# Create symlinks for all the specific .so.8 names that get requested
symlink_targets = [
    '/usr/lib/x86_64-linux-gnu/libcudnn_ops_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_cnn_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_cnn_train.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_ops_train.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_adv_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_adv_train.so.8',
]

for target in symlink_targets:
    if not os.path.exists(target):
        result = subprocess.run(
            ['ln', '-sf', cudnn8_source, target],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"✓ Created: {os.path.basename(target)}")
        else:
            print(f"✗ Failed: {target} — {result.stderr}")
    else:
        print(f"✓ Already exists: {os.path.basename(target)}")

# Update the dynamic linker cache
subprocess.run(['ldconfig'], capture_output=True)
print("\n✓ Library cache updated")
print("cuDNN 8 ops now available to bitsandbytes and other libraries")

✓ Created: libcudnn_ops_infer.so.8
✓ Created: libcudnn_cnn_infer.so.8
✓ Created: libcudnn_cnn_train.so.8
✓ Created: libcudnn_ops_train.so.8
✓ Created: libcudnn_adv_infer.so.8
✓ Created: libcudnn_adv_train.so.8

✓ Library cache updated
cuDNN 8 ops now available to bitsandbytes and other libraries


In [ ]:
import torch
print(f"CUDA initialized: {torch.cuda.is_initialized()}")

CUDA initialized: False


Cell 6 — Download model weights to local disk

In [ ]:
!pip install -q huggingface_hub

In [ ]:
import os
import subprocess

# ── WHISPER ───────────────────────────────────────────────────────────────
# Check for actual weight file, not just directory existence
# 4.0K directory = empty = git lfs didn't download weights
whisper_model_bin = f"{local_paths['whisper']}/model.bin"
whisper_ok = os.path.exists(whisper_model_bin) and os.path.getsize(whisper_model_bin) > 1e9

if not whisper_ok:
    print("Downloading Whisper (~1.5GB)...")
    if os.path.exists(local_paths['whisper']):
        import shutil
        shutil.rmtree(local_paths['whisper'])
    !git lfs install
    !git clone https://huggingface.co/Systran/faster-distil-whisper-large-v3 \
        {local_paths['whisper']}
else:
    print("✓ Whisper already on local disk with real weights")

# Verify
size_gb = os.path.getsize(whisper_model_bin) / 1e9
print(f"  model.bin: {size_gb:.2f} GB {'✓' if size_gb > 1 else '✗ pointer file!'}")

# # ── MINICPM-V ─────────────────────────────────────────────────────────────
# # MiniCPM-V-2_6-int4 splits weights across two shard files
# # minicpm_shard = f"{local_paths['minicpm']}/model-00001-of-00002.bin"
# minicpm_shard1 = f"{local_paths['minicpm']}/model-00001-of-00008.safetensors"
# minicpm_shard2 = f"{local_paths['minicpm']}/model-00008-of-00008.safetensors"
# minicpm_ok = (os.path.exists(minicpm_shard1) and
#               os.path.getsize(minicpm_shard1) > 1e9 and
#               os.path.exists(minicpm_shard2) and
#               os.path.getsize(minicpm_shard2) > 1e9)

# if not minicpm_ok:
#     print("\nDownloading MiniCPM-V full model (~16GB, one-time)...")
#     if os.path.exists(local_paths['minicpm']):
#         import shutil
#         shutil.rmtree(local_paths['minicpm'])
#     !git lfs install
#     !git clone https://huggingface.co/openbmb/MiniCPM-V-2_6 \
#         {local_paths['minicpm']}
# else:
#     print("\n✓ MiniCPM-V already on local disk with real weights")

# # Verify both shards
# # Check first shard exists and is real
# for shard in ['model-00001-of-00008.safetensors']:
#     shard_path = f"{local_paths['minicpm']}/{shard}"
#     if os.path.exists(shard_path):
#         size_gb = os.path.getsize(shard_path) / 1e9
#         print(f"  {shard}: {size_gb:.2f} GB "
#               f"{'✓' if size_gb > 1 else '✗ pointer file!'}")
#     else:
#         print(f"  {shard}: ✗ NOT FOUND")

# ── MINICPM-V ─────────────────────────────────────────────────────────────

from huggingface_hub import login, snapshot_download
import os
import shutil

# LOGIN FIRST
# Paste your HF token when prompted
login()

# Expected shard files
minicpm_shard1 = f"{local_paths['minicpm']}/model-00001-of-00004.safetensors"
minicpm_shard4 = f"{local_paths['minicpm']}/model-00004-of-00004.safetensors"
# previously assumed 8 shards, but its actually 4 shards only.
# Verify existing download
minicpm_ok = (
    os.path.exists(minicpm_shard1) and
    os.path.getsize(minicpm_shard1) > 1e9 and
    os.path.exists(minicpm_shard4) and
    os.path.getsize(minicpm_shard4) > 1e9
)

if not minicpm_ok:

    print("\nDownloading MiniCPM-V full model (~16GB, one-time)...")

    # Remove broken partial download
    if os.path.exists(local_paths['minicpm']):
        shutil.rmtree(local_paths['minicpm'])

    # Download model
    snapshot_download(
        repo_id="openbmb/MiniCPM-V-2_6",
        local_dir=local_paths['minicpm'],
        local_dir_use_symlinks=False,
        resume_download=True
    )

else:
    print("\n✓ MiniCPM-V already on local disk with real weights")

# Verify shards
for shard in [
    'model-00001-of-00004.safetensors',
    'model-00004-of-00004.safetensors'
]:
    shard_path = f"{local_paths['minicpm']}/{shard}"

    if os.path.exists(shard_path):
        size_gb = os.path.getsize(shard_path) / 1e9

        print(
            f"  {shard}: {size_gb:.2f} GB "
            f"{'✓' if size_gb > 1 else '✗ pointer/small file!'}"
        )
    else:
        print(f"  {shard}: ✗ NOT FOUND")

# ── IMAGEBIND ─────────────────────────────────────────────────────────────
# ImageBind is a single .pth file downloaded via wget - reliable
imagebind_ok = os.path.exists(local_paths['imagebind']) and \
               os.path.getsize(local_paths['imagebind']) > 1e9

if not imagebind_ok:
    print("\nDownloading ImageBind (~2GB)...")
    !wget -q https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth \
        -O {local_paths['imagebind']}
else:
    print("\n✓ ImageBind already on local disk")

size_gb = os.path.getsize(local_paths['imagebind']) / 1e9
print(f"  imagebind_huge.pth: {size_gb:.2f} GB {'✓' if size_gb > 1 else '✗ too small!'}")

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────
print("\n=== Model verification summary ===")
checks = [
    ("Whisper",    whisper_model_bin,  1e9),
    ("ImageBind",  local_paths['imagebind'], 1e9),
    ("MiniCPM shard1", minicpm_shard1, 1e9),
]
all_ok = True
for name, path, min_size in checks:
    ok = os.path.exists(path) and os.path.getsize(path) > min_size
    print(f"  {'✓' if ok else '✗'} {name}")
    if not ok:
        all_ok = False

if all_ok:
    print("\n✓ All models ready. Safe to continue.")
else:
    raise RuntimeError("✗ Some models missing or incomplete - check output above")

✓ Whisper already on local disk with real weights
  model.bin: 1.51 GB ✓



✓ MiniCPM-V already on local disk with real weights
  model-00001-of-00004.safetensors: 4.87 GB ✓
  model-00004-of-00004.safetensors: 2.06 GB ✓

✓ ImageBind already on local disk
  imagebind_huge.pth: 4.80 GB ✓

=== Model verification summary ===
  ✓ Whisper
  ✓ ImageBind
  ✓ MiniCPM shard1

✓ All models ready. Safe to continue.


In [ ]:
import torch
print(f"CUDA initialized: {torch.cuda.is_initialized()}")

CUDA initialized: False


Cell 7 — Symlinks only, no clone, correct branch

In [ ]:
import os

# Symlink models from local disk into paths the code expects
# The VideoRAG code has hardcoded relative paths for these models
# Symlinks make local disk files appear where the code looks

links = [
    (local_paths['whisper'],
     '/content/VideoRAG/VideoRAG-algorithm/faster-distil-whisper-large-v3'),
    (local_paths['minicpm'],
     '/content/VideoRAG/VideoRAG-algorithm/MiniCPM-V-2_6-int4'),
]

for src, dst in links:
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"✓ Symlink created: {dst}")
    else:
        print(f"✓ Symlink already exists: {dst}")

# ImageBind expects a .checkpoints folder specifically
os.makedirs('/content/VideoRAG/VideoRAG-algorithm/.checkpoints', exist_ok=True)
imagebind_link = '/content/VideoRAG/VideoRAG-algorithm/.checkpoints/imagebind_huge.pth'
if not os.path.exists(imagebind_link):
    os.symlink(local_paths['imagebind'], imagebind_link)
    print(f"✓ Symlink created: {imagebind_link}")
else:
    print(f"✓ Symlink already exists: {imagebind_link}")

# Verify all symlinks resolve to actual files on local disk
# os.path.exists follows the symlink and checks the target exists
print("\n=== Symlink verification ===")
for name, path in [("Whisper",   links[0][1]),
                    ("MiniCPM-V", links[1][1]),
                    ("ImageBind", imagebind_link)]:
    exists = os.path.exists(path)
    print(f"{'✓' if exists else '✗'} {name}: {'accessible' if exists else 'BROKEN - target not found'}")

✓ Symlink created: /content/VideoRAG/VideoRAG-algorithm/faster-distil-whisper-large-v3
✓ Symlink created: /content/VideoRAG/VideoRAG-algorithm/MiniCPM-V-2_6-int4
✓ Symlink created: /content/VideoRAG/VideoRAG-algorithm/.checkpoints/imagebind_huge.pth

=== Symlink verification ===
✓ Whisper: accessible
✓ MiniCPM-V: accessible
✓ ImageBind: accessible


Cell 8 — Download and trim videos directly to Drive

In [ ]:
videos = {
    "3b1b_nn_1_neurons": "https://www.youtube.com/watch?v=aircAruvnKk",
    "3b1b_nn_2_gradient": "https://www.youtube.com/watch?v=IHZwWFHWa-w",
    "3b1b_nn_3_backprop": "https://www.youtube.com/watch?v=Ilg3gGewQ5U",
    "3b1b_nn_4_backprop2": "https://www.youtube.com/watch?v=tIeHLnjs5U8",
}

# How much of each video to use for indexing
# 4 minutes gives enough content for interesting entity extraction
# without burning too many compute units
TRIM_SECONDS = 240  # 4 minutes

from moviepy.video.io.VideoFileClip import VideoFileClip
import os

for name, url in videos.items():
    trimmed_path = f"{drive_paths['trimmed_videos']}/{name}.mp4"
    raw_path = f"{local_paths['raw_videos']}/{name}_raw.mp4"

    if os.path.exists(trimmed_path):
        print(f"✓ {name} already on Drive, skipping")
        continue

    # Step 1: download full video to local disk
    print(f"Downloading {name}...")
    !yt-dlp -o "{raw_path}" \
            --format "bestvideo[ext=mp4]+bestaudio/best[ext=mp4]/best" \
            --merge-output-format mp4 \
            "{url}" -q

    # Step 2: trim in memory, write trimmed version to Drive
    print(f"Trimming to {TRIM_SECONDS//60} minutes...")
    with VideoFileClip(raw_path) as clip:
        duration = min(TRIM_SECONDS, clip.duration)
        trimmed = clip.subclip(0, duration)
        trimmed.write_videofile(
            trimmed_path,
            codec='libx264',
            verbose=False,
            logger=None
        )

    # Step 3: delete the raw file from local disk immediately
    # we only needed it temporarily to trim from
    os.remove(raw_path)
    size_mb = os.path.getsize(trimmed_path) / 1e6
    print(f"✓ {name}: {duration/60:.1f} mins, {size_mb:.0f}MB saved to Drive\n")

print("\n=== Videos on Drive ===")
import glob
for f in sorted(glob.glob(f"{drive_paths['trimmed_videos']}/*.mp4")):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f)}: {size_mb:.0f}MB")

✓ 3b1b_nn_1_neurons already on Drive, skipping
✓ 3b1b_nn_2_gradient already on Drive, skipping
✓ 3b1b_nn_3_backprop already on Drive, skipping
✓ 3b1b_nn_4_backprop2 already on Drive, skipping

=== Videos on Drive ===
  3b1b_nn_1_neurons.mp4: 25MB
  3b1b_nn_2_gradient.mp4: 33MB
  3b1b_nn_3_backprop.mp4: 41MB
  3b1b_nn_4_backprop2.mp4: 11MB


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)


The raw download goes to local disk, gets trimmed immediately, only the trimmed version touches Drive. Raw file is deleted right after trimming. Clean.

Cell 9 — Set API key and configure VideoRAG

In [ ]:
import nest_asyncio
nest_asyncio.apply()
from google.colab import userdata

import os
import sys
import httpx
import asyncio
import types
import torch

sys.path.insert(0, '/content/VideoRAG/VideoRAG-algorithm')

# ── CRITICAL: verify CUDA is not initialized ──────────────────────────────
# MiniCPM-V captioning runs in a forked subprocess
# If CUDA is initialized in parent, fork fails with:
# "Cannot re-initialize CUDA in forked subprocess"
if torch.cuda.is_initialized():
    raise RuntimeError(
        "CUDA already initialized - restart runtime and do NOT call any "
        "torch.cuda functions before this cell. Check Cell 5a and remove "
        "any cuda calls."
    )
print("✓ CUDA not initialized - safe for multiprocessing fork")

# ── pytorchvideo patch ────────────────────────────────────────────────────
import torchvision.transforms.functional as F
fake_module = types.ModuleType('torchvision.transforms.functional_tensor')
for func_name in ['rgb_to_grayscale','adjust_brightness','adjust_contrast',
                  'adjust_saturation','adjust_hue']:
    if hasattr(F, func_name):
        setattr(fake_module, func_name, getattr(F, func_name))
sys.modules['torchvision.transforms.functional_tensor'] = fake_module
print("✓ pytorchvideo patch applied")

# ── clear cached modules ──────────────────────────────────────────────────
mods_to_remove = [key for key in sys.modules if 'videorag' in key]
for mod in mods_to_remove:
    del sys.modules[mod]

# ── API keys ──────────────────────────────────────────────────────────────
# os.environ["DEEPSEEK_API_KEY"] = "your-deepseek-key-here"
os.environ["DEEPSEEK_API_KEY"] = userdata.get('DS_TOKEN')

print("\n=== API Key Format Check ===")
key = os.environ.get("DEEPSEEK_API_KEY", "")
if key.startswith("sk-"):
    print(f"✓ DEEPSEEK_API_KEY: {key[:6]}...{key[-4:]}")
else:
    raise ValueError("✗ DEEPSEEK_API_KEY looks wrong")

# ── Test DeepSeek ─────────────────────────────────────────────────────────
print("\n=== Testing DeepSeek API ===")
async def test_deepseek():
    async with httpx.AsyncClient() as client:
        response = await client.post(
            "https://api.deepseek.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}",
                     "Content-Type": "application/json"},
            json={"model": "deepseek-chat",
                  "messages": [{"role": "user", "content": "Reply with one word: working"}],
                  "max_tokens": 10},
            timeout=30.0
        )
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

result = await test_deepseek()
print(f"✓ DeepSeek response: {result}")

# ── Import config ─────────────────────────────────────────────────────────
# DO NOT call get_bge_model() here - initializes CUDA
# bge-m3 loads lazily after all forked processes complete
from videorag._llm import deepseek_bge_config
from videorag import VideoRAG, QueryParam
print("\n✓ deepseek_bge_config imported")
print("✓ bge-m3 will load lazily after captioning subprocess completes")

# ── Set paths ─────────────────────────────────────────────────────────────
workdir = drive_paths['workdir']
video_paths = [
    f"{drive_paths['trimmed_videos']}/3b1b_nn_1_neurons.mp4",
    f"{drive_paths['trimmed_videos']}/3b1b_nn_2_gradient.mp4",
    f"{drive_paths['trimmed_videos']}/3b1b_nn_3_backprop.mp4",
    f"{drive_paths['trimmed_videos']}/3b1b_nn_4_backprop2.mp4",
]

print("\n=== Input Videos ===")
all_present = True
for vp in video_paths:
    exists = os.path.exists(vp)
    size_mb = os.path.getsize(vp) / 1e6 if exists else 0
    print(f"{'✓' if exists else '✗'} {os.path.basename(vp)}: {size_mb:.0f}MB")
    if not exists:
        all_present = False

if not all_present:
    raise FileNotFoundError("Some videos missing - run Cell 8 first")

print("\n✓ All checks passed. Safe to run Cell 10.")

✓ CUDA not initialized - safe for multiprocessing fork
✓ pytorchvideo patch applied

=== API Key Format Check ===
✓ DEEPSEEK_API_KEY: sk-758...21f6

=== Testing DeepSeek API ===


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


✓ DeepSeek response: working


0it [00:00, ?it/s]

  warnings.warn(

  warnings.warn(

  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)




✓ deepseek_bge_config imported
✓ bge-m3 will load lazily after captioning subprocess completes

=== Input Videos ===
✓ 3b1b_nn_1_neurons.mp4: 25MB
✓ 3b1b_nn_2_gradient.mp4: 33MB
✓ 3b1b_nn_3_backprop.mp4: 41MB
✓ 3b1b_nn_4_backprop2.mp4: 11MB

✓ All checks passed. Safe to run Cell 10.


Cell 9.5 CUDA guard

In [ ]:
import torch
import gc

gc.collect()

# Final gate before indexing
# If CUDA is initialized here, the captioning subprocess will crash
if torch.cuda.is_initialized():
    print("⚠ WARNING: CUDA initialized - captioning fork will fail")
    print("  Restart runtime and check that no torch.cuda calls happen before Cell 10")
else:
    print("✓ CUDA clean - safe to run Cell 10")
    print("  bge-m3 will initialize CUDA only after all subprocess stages complete")

✓ CUDA clean - safe to run Cell 10
  bge-m3 will initialize CUDA only after all subprocess stages complete


In [ ]:
# Phase 6: index and analysis outputs live on /content/, NOT Drive.
# Drive remains read-only source-of-truth for the trimmed videos.
import os
workdir = '/content/videorag_workdir'
analysis_dir = '/content/videorag_analysis'
os.makedirs(workdir, exist_ok=True)
os.makedirs(analysis_dir, exist_ok=True)
print(f'Index workdir: {workdir}')
print(f'Analysis dir:  {analysis_dir}')


Cell 10 — Run indexing

Cell 11 — Inspect workdir structure
This is your first look at what was actually produced.

In [ ]:
# import os
# import json

# # What does the symlink point to?
# link = '/content/VideoRAG/VideoRAG-algorithm/MiniCPM-V-2_6-int4'
# print(f"Symlink points to: {os.path.realpath(link)}")

# # What config does it have?
# config_path = f"{local_paths['minicpm']}/config.json"
# with open(config_path) as f:
#     config = json.load(f)

# print(f"\nModel type: {config.get('model_type')}")
# print(f"Quantization: {config.get('quantization_config', {}).get('quant_method', 'NONE - full precision')}")
# print(f"Transformers version: {config.get('transformers_version')}")

# # List large files
# print("\nLarge files in model folder:")
# for f in os.listdir(local_paths['minicpm']):
#     fp = os.path.join(local_paths['minicpm'], f)
#     if os.path.isfile(fp):
#         size = os.path.getsize(fp) / 1e9
#         if size > 0.1:
#             print(f"  {f}: {size:.2f} GB")

In [ ]:
import os, json, shutil

# Check what's actually on disk
config_path = f"{local_paths['minicpm']}/config.json"
needs_redownload = True
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    quant = config.get('quantization_config', None)
    if quant is None:
        print("✓ Full precision model already present, no redownload needed")
        needs_redownload = False
    else:
        print(f"✗ Quantized model present ({quant.get('quant_method', '?')}), need full precision")
else:
    print("✗ No config.json found, need download")

# if needs_redownload:
#     if os.path.exists(local_paths['minicpm']):
#         print("Deleting old model...")
#         shutil.rmtree(local_paths['minicpm'])
#     print("Downloading full precision MiniCPM-V-2_6 (~16GB, 15-20 min)...")
#     !git lfs install -q
#     !git clone https://huggingface.co/openbmb/MiniCPM-V-2_6 {local_paths['minicpm']}

#     # Verify it's actually full precision now
#     with open(f"{local_paths['minicpm']}/config.json") as f:
#         config = json.load(f)
#     assert config.get('quantization_config') is None, "Still quantized after download — something went wrong"
#     print("✓ Full precision confirmed")

✓ Full precision model already present, no redownload needed


In [ ]:
import time

start_time = time.time()

print("=== Initializing VideoRAG ===")
print(f"Config: deepseek_bge_config")
print(f"  LLM: deepseek-chat (entity extraction, filtering, generation)")
print(f"  Embeddings: BAAI/bge-m3 local on A100 (dim=1024)")
print(f"Workdir: {workdir}\n")

videorag = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=workdir,
    analysis_output_dir=analysis_dir,
)

print("=== Starting Indexing Pipeline ===")
print("Stage 1: Split videos into 30s clips")
print("Stage 2: ASR - Whisper transcribes each clip")
print("Stage 3: VLM - MiniCPM-V captions each clip")
print("Stage 4: LLM - DeepSeek extracts entities + relationships per chunk")
print("Stage 5: LLM - DeepSeek merges and synthesizes entity descriptions")
print("Stage 6: Embeddings - bge-m3 embeds chunks and entities on GPU")
print("Stage 7: ImageBind embeds each clip visually")
print("\nWatch the output below for stage transitions...\n")

videorag.insert_video(video_path_list=video_paths)

elapsed = time.time() - start_time
print(f"\n✓ Indexing complete in {elapsed/60:.1f} minutes")
print(f"Index saved to: {workdir}")

=== Initializing VideoRAG ===
Config: deepseek_bge_config
  LLM: deepseek-chat (entity extraction, filtering, generation)
  Embeddings: BAAI/bge-m3 local on A100 (dim=1024)
Workdir: /content/drive/MyDrive/videorag_project/workdir

=== Starting Indexing Pipeline ===
Stage 1: Split videos into 30s clips
Stage 2: ASR - Whisper transcribes each clip
Stage 3: VLM - MiniCPM-V captions each clip
Stage 4: LLM - DeepSeek extracts entities + relationships per chunk
Stage 5: LLM - DeepSeek merges and synthesizes entity descriptions
Stage 6: Embeddings - bge-m3 embeds chunks and entities on GPU
Stage 7: ImageBind embeds each clip visually

Watch the output below for stage transitions...



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Captioning Video 3b1b_nn_1_neurons:   0%|          | 0/8 [00:00<?, ?it/s]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(

Encoding Video Segments 3b1b_nn_4_backprop2: 100%|██████████| 4/4 [01:27<00:00, 22.00s/it]


Loading bge-m3 onto GPU...


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

✓ bge-m3 loaded


✓ Indexing complete in 29.7 minutes
Index saved to: /content/drive/MyDrive/videorag_project/workdir


In [ ]:
# Phase 6: post-index extraction. Reads workdir (no model calls), writes
# indexing analysis JSONs into analysis_dir/indexing/.
%cd /content/VideoRAG/VideoRAG-algorithm
!python scripts/extract_indexing_analysis.py \
    --workdir {workdir} \
    --output-dir {analysis_dir}/indexing \
    --config-name deepseek_bge_config

import os
indexing_dir = f'{analysis_dir}/indexing'
print('\nExtraction outputs:')
for name in sorted(os.listdir(indexing_dir)):
    size_kb = os.path.getsize(f'{indexing_dir}/{name}') / 1e3
    print(f'  {name} ({size_kb:.1f} KB)')


Setup before inspection - switch to cpu now, everything in index is saved to drive already

In [ ]:
# === Setup for inspection (CPU runtime, no GPU needed) ===
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json

# Paths (same as indexing session)
drive_root = '/content/drive/MyDrive/videorag_project'
drive_paths = {
    'trimmed_videos': f'{drive_root}/trimmed_videos',
    'workdir':        f'{drive_root}/workdir',
    'inspection':     f'{drive_root}/inspection_outputs',
}
workdir = drive_paths['workdir']

os.makedirs(drive_paths['inspection'], exist_ok=True)

# Verify workdir has content
files = os.listdir(workdir)
print(f"Workdir: {workdir}")
print(f"Files: {len(files)}")
for f in sorted(files):
    size_kb = os.path.getsize(os.path.join(workdir, f)) / 1e3
    print(f"  {f} ({size_kb:.1f} KB)")

print(f"\n✓ Ready for inspection — no GPU, no repo clone, no models needed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workdir: /content/drive/MyDrive/videorag_project/workdir
Files: 17
  3b1b_nn_1_neurons_captions.json (9.4 KB)
  3b1b_nn_1_neurons_transcripts.json (5.0 KB)
  3b1b_nn_2_gradient_captions.json (10.8 KB)
  3b1b_nn_2_gradient_transcripts.json (5.1 KB)
  3b1b_nn_3_backprop_captions.json (12.9 KB)
  3b1b_nn_3_backprop_transcripts.json (5.6 KB)
  3b1b_nn_4_backprop2_captions.json (9.3 KB)
  3b1b_nn_4_backprop2_transcripts.json (4.9 KB)
  _cache (4.1 KB)
  graph_chunk_entity_relation.graphml (137.9 KB)
  kv_store_llm_response_cache.json (143.6 KB)
  kv_store_text_chunks.json (66.4 KB)
  kv_store_video_path.json (0.4 KB)
  kv_store_video_segments.json (89.7 KB)
  vdb_chunks.json (88.3 KB)
  vdb_entities.json (954.8 KB)
  vdb_video_segment_feature.json (177.8 KB)

✓ Ready for inspection — no GPU, no repo clone, no models needed


In [ ]:
import os
import json

print("=== WORKDIR STRUCTURE ===\n")

# Check workdir actually has content
contents = list(os.walk(workdir))
if len(contents) <= 1 and not os.listdir(workdir):
    raise RuntimeError("Workdir is empty - indexing may not have completed. Check Cell 10 output.")

for root, dirs, files in os.walk(workdir):
    dirs[:] = [d for d in dirs if d != '_cache']
    level = root.replace(workdir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        filepath = os.path.join(root, file)
        size_kb = os.path.getsize(filepath) / 1e3
        print(f"{indent}  {file} ({size_kb:.1f} KB)")

cache_dir = os.path.join(workdir, '_cache')
if os.path.exists(cache_dir):
    mp3_files = []
    mp4_files = []
    for root, dirs, files in os.walk(cache_dir):
        for f in files:
            if f.endswith('.mp3'): mp3_files.append(f)
            if f.endswith('.mp4'): mp4_files.append(f)
    print(f"\n_cache/ (shown separately)")
    print(f"  Audio clips (.mp3): {len(mp3_files)}")
    print(f"  Video clips (.mp4): {len(mp4_files)}")
    print(f"  Total clips: {len(mp3_files)} (each is one 30-second segment)")
    print(f"  Expected for 16 mins of video: ~32 clips")

=== WORKDIR STRUCTURE ===

workdir/
  3b1b_nn_1_neurons_transcripts.json (5.0 KB)
  3b1b_nn_1_neurons_captions.json (9.4 KB)
  3b1b_nn_2_gradient_transcripts.json (5.1 KB)
  3b1b_nn_2_gradient_captions.json (10.8 KB)
  3b1b_nn_3_backprop_transcripts.json (5.6 KB)
  3b1b_nn_3_backprop_captions.json (12.9 KB)
  3b1b_nn_4_backprop2_transcripts.json (4.9 KB)
  3b1b_nn_4_backprop2_captions.json (9.3 KB)
  kv_store_text_chunks.json (66.4 KB)
  vdb_entities.json (954.8 KB)
  vdb_chunks.json (88.3 KB)
  graph_chunk_entity_relation.graphml (137.9 KB)
  vdb_video_segment_feature.json (177.8 KB)
  kv_store_video_segments.json (89.7 KB)
  kv_store_video_path.json (0.4 KB)
  kv_store_llm_response_cache.json (143.6 KB)

_cache/ (shown separately)
  Audio clips (.mp3): 0
  Video clips (.mp4): 0
  Total clips: 0 (each is one 30-second segment)
  Expected for 16 mins of video: ~32 clips


Cell 12 — Read and display transcripts

In [ ]:
import json
import os

print("=== TRANSCRIPTS AND CAPTIONS PER CLIP ===\n")
print("Raw output of ASR (Whisper) + VLM (MiniCPM-V)")
print("For each 30-second clip: what was heard + what was seen\n")

chunk_files = []
for root, dirs, files in os.walk(workdir):
    for f in files:
        if 'chunk' in f.lower() or 'caption' in f.lower() or 'text' in f.lower():
            chunk_files.append(os.path.join(root, f))

if not chunk_files:
    print("No chunk/caption files found yet.")
    print("These files are written during indexing.")
    print("Run this cell after Cell 10 completes.")
else:
    all_data = {}
    for cf in chunk_files[:3]:
        print(f"\n--- File: {os.path.basename(cf)} ---")
        with open(cf) as fp:
            data = json.load(fp)
        all_data[os.path.basename(cf)] = data

        if isinstance(data, dict):
            print(f"Type: dict with {len(data)} entries")
            for i, (key, value) in enumerate(data.items()):
                if i >= 2: break
                print(f"\nEntry key: {key}")
                print(json.dumps(value, indent=2)[:800])
                print("...")
        elif isinstance(data, list):
            print(f"Type: list with {len(data)} entries")
            for item in data[:2]:
                print(json.dumps(item, indent=2)[:800])
                print("...")

    # Save actual data, not a placeholder
    inspection_file = f"{drive_paths['inspection']}/transcripts_and_captions.json"
    with open(inspection_file, 'w') as f:
        json.dump(all_data, f, indent=2)
    print(f"\n✓ Actual data saved to Drive: {inspection_file}")

=== TRANSCRIPTS AND CAPTIONS PER CLIP ===

Raw output of ASR (Whisper) + VLM (MiniCPM-V)
For each 30-second clip: what was heard + what was seen


--- File: 3b1b_nn_1_neurons_captions.json ---
Type: dict with 8 entries

Entry key: 0
"The video begins with a black screen and transitions to an animated character resembling the mathematical symbol pi (\u03c0) in blue, standing against a dark background. A large white \"3\" is displayed next to the character, suggesting a connection between the two elements. The character appears surprised or confused as it looks at the number. As the scene progresses, a thought bubble emerges from the character's head containing the same \"3\", indicating that the character is thinking about this number.Suddenly, the number changes to \"3\u21923\", hinting at a transformation or repetition of the digit within the character's thought process. The word \"How?!!?\" is added beneath the brain image inside the thought bubble, expressing amazement or disbelief.

Cell 13 — Read and display raw entity/relationship extraction per chunk

In [ ]:
import json
import os
import xml.etree.ElementTree as ET

print("=== ENTITIES AND RELATIONSHIPS ===\n")

# ── Part 1: Entity embeddings (vdb_entities.json) ──
vdb_path = os.path.join(workdir, 'vdb_entities.json')
with open(vdb_path) as f:
    vdb = json.load(f)

entities = vdb.get('data', [])
print(f"Total entities: {len(entities)}")
print(f"Embedding dim: {vdb.get('embedding_dim')}\n")

print("Sample entities (first 10):")
for ent in entities[:10]:
    print(f"  {ent.get('entity_name', '?')}")

# ── Part 2: Knowledge graph (GraphML) ──
graph_path = os.path.join(workdir, 'graph_chunk_entity_relation.graphml')
tree = ET.parse(graph_path)
root = tree.getroot()

# GraphML uses a namespace
ns = {'g': 'http://graphml.graphstruct.org/graphml'}
# Try auto-detect namespace
for elem in root.iter():
    if '}' in elem.tag:
        ns['g'] = elem.tag.split('}')[0].strip('{')
        break

nodes = root.findall('.//g:node', ns) or root.findall('.//{http://graphml.graphstruct.org/graphml}node') or root.findall('.//node')
edges = root.findall('.//g:edge', ns) or root.findall('.//{http://graphml.graphstruct.org/graphml}edge') or root.findall('.//edge')

print(f"\n=== KNOWLEDGE GRAPH ===")
print(f"Nodes (entities): {len(nodes)}")
print(f"Edges (relationships): {len(edges)}\n")

# Show first 5 nodes with their data
print("Sample nodes:")
for node in nodes[:5]:
    node_id = node.get('id', '?')
    # Extract data fields from <data> child elements
    fields = {d.get('key', '?'): (d.text or '')[:150] for d in node}
    print(f"\n  Node: {node_id}")
    for k, v in fields.items():
        print(f"    {k}: {v}")

print("\nSample edges:")
for edge in edges[:5]:
    src = edge.get('source', '?')
    tgt = edge.get('target', '?')
    fields = {d.get('key', '?'): (d.text or '')[:150] for d in edge}
    print(f"\n  {src} → {tgt}")
    for k, v in fields.items():
        print(f"    {k}: {v}")

=== ENTITIES AND RELATIONSHIPS ===

Total entities: 172
Embedding dim: 1024

Sample entities (first 10):
  "Π"
  "3"
  "VISUAL CORTEX"
  "EYE"
  "28 BY 28 PIXEL GRID"
  "MACHINE LEARNING"
  "NEURAL NETWORK"
  "HANDWRITTEN DIGITS"
  "THE VIDEO"
  "THE SPEAKER"

=== KNOWLEDGE GRAPH ===
Nodes (entities): 173
Edges (relationships): 200

Sample nodes:

  Node: "Π"
    d0: "PERSON"
    d1: "π is an animated anthropomorphic character resembling the mathematical symbol pi, shown with expressive features and cognitive processes involving nu
    d2: chunk-c62b0e752b297645334ab25a3bc92a20

  Node: "3"
    d0: "EVENT"
    d1: "3 is a numeric concept repeatedly displayed and recognized in various forms, central to the video's theme of number recognition."
    d2: chunk-c62b0e752b297645334ab25a3bc92a20

  Node: "VISUAL CORTEX"
    d0: "ORGANIZATION"
    d1: "Visual Cortex is a part of the brain mentioned in the transcript, described as capable of resolving different visual inputs as representing the

Cell 14 — Visualize the merged knowledge graph

In [ ]:
import networkx as nx
import json

# Load the real graph file
graph_path = f"{workdir}/graph_chunk_entity_relation.graphml"
G = nx.read_graphml(graph_path)

print(f"Nodes (entities): {G.number_of_nodes()}")
print(f"Edges (relationships): {G.number_of_edges()}")

# Show all entities with descriptions
print(f"\n{'='*60}")
print("ENTITIES")
print('='*60)
for node_id, attrs in G.nodes(data=True):
    desc = attrs.get('description', '')[:300]
    source = attrs.get('source_id', '')[:200]
    print(f"\n  {node_id}")
    print(f"  Description: {desc}")
    print(f"  Source: {source}")

# Show all relationships
print(f"\n{'='*60}")
print("RELATIONSHIPS")
print('='*60)
for src, tgt, attrs in G.edges(data=True):
    desc = attrs.get('description', '')[:200]
    print(f"\n  {src}  →  {tgt}")
    print(f"  {desc}")

# Find cross-video entities (merged across videos)
print(f"\n{'='*60}")
print("CROSS-VIDEO ENTITIES (appeared in multiple videos)")
print('='*60)
for node_id, attrs in G.nodes(data=True):
    source = attrs.get('source_id', '')
    videos = set()
    for part in source.split('<SEP>'):
        # Extract video name from chunk IDs
        for token in part.split(','):
            token = token.strip()
            if token.startswith('3b1b_'):
                vid = '_'.join(token.split('_')[:5])  # e.g. 3b1b_nn_1_neurons
                videos.add(vid)
    if len(videos) > 1:
        print(f"\n  {node_id} — appears in {len(videos)} videos:")
        for v in sorted(videos):
            print(f"    • {v}")

# Save the real graph data to Drive
graph_data = {
    "stats": {
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges()
    },
    "entities": {n: dict(a) for n, a in G.nodes(data=True)},
    "relationships": [
        {"source": s, "target": t, **dict(a)}
        for s, t, a in G.edges(data=True)
    ]
}
with open(f"{drive_paths['inspection']}/merged_graph_real.json", 'w') as f:
    json.dump(graph_data, f, indent=2)
print(f"\n✓ Real graph data saved to Drive: merged_graph_real.json")

Nodes (entities): 173
Edges (relationships): 200

ENTITIES

  "Π"
  Description: "π is an animated anthropomorphic character resembling the mathematical symbol pi, shown with expressive features and cognitive processes involving numbers."
  Source: chunk-c62b0e752b297645334ab25a3bc92a20

  "3"
  Description: "3 is a numeric concept repeatedly displayed and recognized in various forms, central to the video's theme of number recognition."
  Source: chunk-c62b0e752b297645334ab25a3bc92a20

  "VISUAL CORTEX"
  Description: "Visual Cortex is a part of the brain mentioned in the transcript, described as capable of resolving different visual inputs as representing the same idea, highlighting its role in perception and cognition."
  Source: chunk-c62b0e752b297645334ab25a3bc92a20

  "EYE"
  Description: "Eye is a sensory organ mentioned in the transcript, containing sensitive cells that fire differently when seeing different representations of the number 3."
  Source: chunk-c62b0e752b297645334ab

Cell 15 — Run a retrieval query

In [ ]:
import os
import sys
import multiprocessing

sys.path.insert(0, '/content/VideoRAG/VideoRAG-algorithm')

# ── API KEYS ──────────────────────────────────────────────────────────────
# Must be set again - environment variables don't persist across sessions
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DS_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from videorag._llm import deepseek_bge_config
from videorag import VideoRAG, QueryParam

query = "How does backpropagation use the chain rule to compute gradients?"

print(f"Query: {query}\n")
print("Retrieval pipeline:")
print("  Path 1: entity matching via bge-m3 → graph provenance → clips")
print("  Path 2: visual scene description → ImageBind → clips")
print("  Path 3: direct chunk retrieval via bge-m3")
print("  Then: intersect Path1+2 → DeepSeek filter → re-caption → generate\n")

# ── LOAD SAVED INDEX ──────────────────────────────────────────────────────
# This does NOT re-index. It loads the graph and embeddings from workdir.
# Workdir is on Drive so it persists from the indexing session.
multiprocessing.set_start_method('spawn', force=True)

workdir = drive_paths['workdir']

videorag_query = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=workdir
)

# Load MiniCPM-V for query-time re-captioning
# This is the only local model needed at retrieval time
videorag_query.load_caption_model(debug=False)

param = QueryParam(mode="videorag")
param.wo_reference = False  # Include video name + timestamps in response

response = videorag_query.query(query=query, param=param)

print("=== RESPONSE ===\n")
print(response)

# Save to Drive
with open(f"{drive_paths['inspection']}/query_response.txt", 'w') as f:
    f.write(f"Query: {query}\n\n")
    f.write(f"Response:\n{response}")
print(f"\n✓ Response saved to Drive: {drive_paths['inspection']}/query_response.txt")

Query: How does backpropagation use the chain rule to compute gradients?

Retrieval pipeline:
  Path 1: entity matching via bge-m3 → graph provenance → clips
  Path 2: visual scene description → ImageBind → clips
  Path 3: direct chunk retrieval via bge-m3
  Then: intersect Path1+2 → DeepSeek filter → re-caption → generate



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The method by which backpropagation uses the chain rule to compute gradients.
Retrieved Text Segments {'3b1b_nn_4_backprop2_4', '3b1b_nn_2_gradient_1', '3b1b_nn_4_backprop2_5', '3b1b_nn_4_backprop2_6', '3b1b_nn_2_gradient_0'}
A scene explaining how backpropagation uses the chain rule to compute gradients.
Retrieved Visual Segments {'3b1b_nn_4_backprop2_5', '3b1b_nn_4_backprop2_7', '3b1b_nn_4_backprop2_6', '3b1b_nn_4_backprop2_3'}
7 Video Segments remain after filtering
Remain segments ['3b1b_nn_2_gradient_0', '3b1b_nn_2_gradient_1', '3b1b_nn_4_backprop2_3', '3b1b_nn_4_backprop2_4', '3b1b_nn_4_backprop2_5', '3b1b_nn_4_backprop2_6', '3b1b_nn_4_backprop2_7']
Keywords: backpropagation, chain rule, compute, gradients


Captioning Segments for Given Query:   0%|          | 0/7 [00:00<?, ?it/s]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(

Captioning Segments for Given Query: 100%|██████████| 7/7 [02:33<00:00, 21.98s/it]


=== RESPONSE ===

Backpropagation is the core algorithm that enables neural networks to learn by efficiently computing the gradient of the cost function with respect to every weight and bias in the network. At its heart, backpropagation relies on the chain rule from calculus to break down this complex computation into a series of manageable steps, propagating the error signal backward from the output layer to the input layer.

### The Core Idea: Sensitivity and the Chain Rule

The fundamental goal is to determine how sensitive the total cost (or error) of the network is to small changes in each individual weight. The chain rule provides a way to calculate this sensitivity by considering the chain of intermediate variables that connect a weight to the final cost [1].

For a simple network with one neuron per layer, the process is as follows. A small nudge to a weight, say \( w^{(L)} \) in the last layer, causes a small change in the weighted sum \( z^{(L)} \), which in turn changes the 

What you now have after all 15 cells
On Drive, in inspection_outputs/:

transcripts_and_captions.json — what Whisper heard and what MiniCPM-V saw, per clip
entity_extraction.json — what GPT-4o-mini extracted per chunk, before merging
merged_graph.json — the final unified graph across all 4 videos
query_response.txt — retrieval output with video timestamps

In workdir/:

The complete hybrid index ready for future retrieval sessions without re-running any models

The inspection cells are written to show you the data at each stage. Once you run indexing and see the actual file names and structures, Cells 12-14 may need minor adjustments — the exact JSON keys depend on what nano-graphrag names things internally. When you run them, tell me what they print and I'll refine the inspection code immediately.
Get the API key set up and run Cell 1 first. Tell me what GPU and RAM it shows.

Cell 16 - other queries to test the framework

In [ ]:
queries = [
    # Multi-hop: requires connecting concepts across video 1 (neurons) and video 3 (backprop)
    "How does the structure of neurons and layers described in the first video "
    "relate to the way backpropagation computes updates in later videos?",

    # Specific single-clip retrieval: the answer lives in one place
    "What analogy or visual example does the narrator use to explain "
    "why the sigmoid function is used as an activation function?",

    # Synthesis across all 4 videos: needs to aggregate a progression
    "Trace the full learning pipeline from how a neural network represents "
    "data through to how it actually updates its weights via gradient descent "
    "and backpropagation.",
]

import time

for i, query in enumerate(queries, 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i}")
    print(f"{'='*70}")
    print(f"{query}\n")

    start = time.time()
    param = QueryParam(mode="videorag")
    param.wo_reference = False

    response = videorag_query.query(query=query, param=param)

    elapsed = time.time() - start
    print(f"\n--- Response ({elapsed:.0f}s) ---\n")
    print(response)
    print(f"\n{'='*70}\n")

    # Save each response
    with open(f"{drive_paths['inspection']}/query_{i}_response.txt", 'w') as f:
        f.write(f"Query: {query}\n\nResponse:\n{response}")

print("✓ All responses saved to Drive")


QUERY 1
How does the structure of neurons and layers described in the first video relate to the way backpropagation computes updates in later videos?

The structure of neurons and layers and how it relates to the way backpropagation computes updates.
Retrieved Text Segments {'3b1b_nn_1_neurons_3', '3b1b_nn_2_gradient_2', '3b1b_nn_2_gradient_3', '3b1b_nn_1_neurons_2'}
The structure of neurons and layers in the first video relates to how backpropagation computes updates in later videos.
Retrieved Visual Segments {'3b1b_nn_1_neurons_5', '3b1b_nn_1_neurons_4', '3b1b_nn_2_gradient_3', '3b1b_nn_2_gradient_0'}
7 Video Segments remain after filtering
Remain segments ['3b1b_nn_1_neurons_2', '3b1b_nn_1_neurons_3', '3b1b_nn_1_neurons_4', '3b1b_nn_1_neurons_5', '3b1b_nn_2_gradient_0', '3b1b_nn_2_gradient_2', '3b1b_nn_2_gradient_3']
Keywords: structure, neurons, layers, first video, backpropagation, computes updates, later videos


Captioning Segments for Given Query: 100%|██████████| 7/7 [02:37<00:00, 22.51s/it]



--- Response (187s) ---

# From Neural Structure to Backpropagation: Connecting the Dots

## The Foundation: Neuron Structure and Layer Architecture

The first video establishes a clear, intuitive foundation for understanding neural networks. At its core, a neuron is described as "a thing that holds a number, specifically a number between zero and one" [1]. This simple definition becomes the building block for the entire network architecture. The network begins with an input layer of 784 neurons, each corresponding to a pixel in a 28×28 grayscale image of a handwritten digit, with activation values ranging from 0 (black) to 1 (white) [2].

The layered structure is motivated by a hierarchical learning approach. The video explains that "maybe the second layer could pick up on the edges, and the third layer might pick up on patterns like loops and lines, and the last one could just piece together those patterns to recognize digits" [3]. This conceptual framework—where each layer builds u

Captioning Segments for Given Query: 100%|██████████| 1/1 [00:21<00:00, 21.54s/it]



--- Response (42s) ---

Based on the retrieved information, the narrator does not use a specific analogy or visual example to explain *why* the sigmoid function is used as an activation function. Instead, the retrieved content focuses on *how* the sigmoid function operates within the network, describing it as a "sigmoid squishification" that compresses the weighted sum into a value between 0 and 1 [1]. The explanation is centered on the mechanics of the function (e.g., a neuron only activates when the weighted sum exceeds a threshold like 10) rather than a metaphorical or visual example for its selection [2].

#### Reference:
[1] 3b1b_nn_2_gradient, 02:20, 06:62  
[2] 3b1b_nn_2_gradient, 00:00, 02:20



QUERY 3
Trace the full learning pipeline from how a neural network represents data through to how it actually updates its weights via gradient descent and backpropagation.

The full learning pipeline from how a neural network represents data through to how it updates its weights via gr

Captioning Segments for Given Query: 100%|██████████| 7/7 [02:33<00:00, 21.92s/it]



--- Response (183s) ---

# The Neural Network Learning Pipeline: From Data Representation to Weight Updates

## Data Representation and Network Architecture

The learning process begins with how data is represented within the neural network. In the classic example of handwritten digit recognition, each image is rendered on a 28×28 pixel grid, where each pixel has a grayscale value between 0 and 1. These values determine the activations of 784 neurons in the input layer of the network [1]. As one video explains, "the network starts with a bunch of neurons corresponding to each of the 28 times 28 pixels of the input image, which is 784 neurons in total" [2]. Each neuron holds a number representing the grayscale value of its corresponding pixel, ranging from 0 for black pixels up to 1 for white pixels. This number inside the neuron is called its activation [2].

The network architecture typically includes multiple layers. For the digit recognition example, a common configuration uses two

In [ ]:
# Phase 6: load eval set. EVAL_SET_PATH defaults to the version committed
# in the repo; override if you uploaded a different one to Drive.
import json
EVAL_SET_PATH = '/content/VideoRAG/notes/eval_set.json'

with open(EVAL_SET_PATH, encoding='utf-8') as f:
    eval_set = json.load(f)

queries = eval_set['queries']
print(f'Loaded {len(queries)} queries from {EVAL_SET_PATH}')
for q in queries:
    txt = q['query_text']
    print(f"  {q['query_id']}: {txt[:70]}{'...' if len(txt) > 70 else ''}")


In [ ]:
# Phase 6: run each eval query with full instrumentation. Each query
# produces its own folder under analysis_dir/queries/<query_id>/.
import multiprocessing, time
from videorag._llm import deepseek_bge_config
from videorag import VideoRAG, QueryParam

multiprocessing.set_start_method('spawn', force=True)

videorag_query = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=workdir,
    analysis_output_dir=analysis_dir,
)
videorag_query.load_caption_model(debug=False)

for q in queries:
    print(f"\n{'='*70}")
    print(f"{q['query_id']}: {q['query_text']}")
    print('='*70)
    param = QueryParam(mode='videorag')
    param.wo_reference = False
    param.query_id = q['query_id']
    param.query_metadata = {
        k: v for k, v in q.items() if k not in ('query_id', 'query_text')
    }
    start = time.time()
    response = videorag_query.query(query=q['query_text'], param=param)
    elapsed = time.time() - start
    print(f'\n--- Response ({elapsed:.0f}s) ---')
    print(response)
    print(f"\nAnalysis folder: {analysis_dir}/queries/{q['query_id']}/")

print(f"\n{'='*70}")
print(f'All {len(queries)} queries done. Outputs in {analysis_dir}/queries/')


In [ ]:
# Phase 6: zip analysis outputs for download.
# The index at /content/videorag_workdir/ is left in place so additional
# queries can re-use it. When the runtime disconnects, both /content/
# locations are wiped — download the zip BEFORE disconnecting.
import os
zip_path = '/content/analysis_outputs.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

!zip -r -q /content/analysis_outputs.zip {analysis_dir}/

size_mb = os.path.getsize(zip_path) / 1e6
print(f'wrote {zip_path} ({size_mb:.1f} MB)')
print("\nDownload via Colab file browser (left sidebar -> Files -> right-click -> Download).")
print(f'\nNote: the index at {workdir} is preserved for re-querying in this session.')


 The structure is a list, not a tuple or dict. video_frames + [query] is Python list concatenation: [PIL_image_1, PIL_image_2, PIL_image_3, PIL_image_4, PIL_image_5, "the transcript+prompt string"]. Six elements: five PIL Images followed by one string. This list becomes the content of a single chat message wearing the user role.

the chat message format is what the .chat() method expects. Specifically:pythonmsgs = [{'role': 'user', 'content': [<PIL.Image>, <PIL.Image>, ..., <str>]}]This is a multimodal message format (similar to OpenAI's vision API). The model's .chat() method is built to handle a content field that's a list of mixed types — PIL Images and strings interleaved in whatever order you want. You could write [str, Image, str, Image, str] and it would still work; the model preserves order.